# ESP32 species classifier v4.1: detector-box crops, visits, automation rate

Retrains **only the species model**; isBird v4 is unchanged. The changes from v4:

1. **Input = a crop around the detector's bird box.** The notebook runs the deployed Pi detector (`pi_detector_float_stride16_224`) over every capture, on the same static-ROI crop the Pi uses, and trains and tests on crops built from **its** boxes. The test therefore measures the chain the hub will run: isBird gate → detector box → species. Hand-labelled boxes and loose crops are mixed in, so a slightly wrong box or a detector miss doesn't break it.
2. **Visits.** Captures from the same camera less than `VISIT_GAP_SECONDS` apart form a visit. Training samples each visit equally (no more 52-frame dove visits dominating). Validation and test hold out whole visits, stratified by species. Results are reported per visit with probabilities averaged across its frames, which is how the hub should label birds.
3. **Training fixes.** The checkpoint is picked on class-balanced validation log-loss over camera species, with a minimum epoch count. Label smoothing is 0.05.
4. **Automation, not accuracy.** The model auto-labels a visit only when its averaged confidence clears a threshold chosen on validation for `AUTO_TARGET_PRECISION`, and only for species with enough camera training visits. Otherwise it falls back to a species *group*, or sends the visit to review. The headline number is **the share of test visits labelled automatically, and how precise those labels are**.
5. **Curation outputs:** `visits.csv` (one row per visit) and `review_queue.csv` (visits where the model confidently disagrees with your label, which are likely label errors or hard cases).


## 1. Configuration


In [1]:
# ---- Data -------------------------------------------------------------------
DATASET_ROOT = r"D:\Datasets\ESP32Cam_confirmed"
ANNOTATIONS_CSV = DATASET_ROOT + r"\bird_annotations.csv"
INVENTORY_CSV = DATASET_ROOT + r"\ESP32Cam_confirmed_manifest.csv"
PROJECT_ROOT = r"C:\Users\Joe\Desktop\ESP_Classifier\esp32_Classifier"
MODELS_DIR = PROJECT_ROOT + r"\Models"
OUTPUT_ROOT = MODELS_DIR + r"\runs"
OUTSIDE_MANIFEST_DIRS = [DATASET_ROOT + r"\Manifests", MODELS_DIR + r"\Manifests"]
INAT_IMAGE_ROOTS = [r"D:\Datasets\iNaturalist", r"D:\Datasets\Animals", r"E:\Datasets\iNaturalist", r"E:\Datasets\Animals"]
CUB_IMAGE_ROOTS = [r"D:\Datasets\CUB_200\CUB_200_2011\images", r"D:\Datasets\CUB_200_2011\CUB_200_2011\images",
                   r"E:\Datasets\CUB_200\CUB_200_2011\images", r"E:\Datasets\CUB_200_2011\CUB_200_2011\images"]
# Localisers: both are run on every capture; LOCALIZER = "auto" keeps whichever finds more labelled birds.
#  - pi_detector:   the hub's current detector, on the Pi-selected crop, decoded by server/bird_inference.py
#  - esp32_roi_v3:  the v3 ESP32 ROI detector (128 px, int8), on a square around the Pi-selected crop
SERVER_DIR = PROJECT_ROOT + r"\server"
DETECTOR_MODEL = MODELS_DIR + r"\pi_detector_float_stride16_224\bird_detector_motion_crop_stride16_float32.tflite"
ROI_V3_DIR = MODELS_DIR + r"\Multi_Object_Detector\runs\esp32_roi_detector_v3_128_20261110T052149Z\selected_model"
LOCALIZER = "auto"                 # "auto", "esp32_roi_v3" or "pi_detector"
DETECTOR_CACHE_DIR = OUTPUT_ROOT + r"\_detector_cache"
PREVIOUS_SPECIES_DIR = MODELS_DIR + r"\runs\esp32_classifiers_v4_20261119T064312Z\species"  # compared on the same crops

SEED = 1234
QUICK_RUN = False
WEIGHTS = "imagenet"

# ---- Visits and splits ----------------------------------------------------------
VISIT_GAP_SECONDS = 120            # same camera, captures closer than this belong to one visit
VAL_VISIT_FRACTION = 0.15          # per species, by visit
TEST_VISIT_FRACTION = 0.20

# ---- Detector pass -------------------------------------------------------------
DETECTOR_MIN_SCORE = 0.05          # isBird is the gate; the detector only has to localise
DETECTOR_MATCH_IOU = 0.30          # a detection belongs to the labelled box it overlaps most, if IoU >= this

# ---- Species crops -------------------------------------------------------------
PI_MARGIN_FRACTION = 0.20          # server/motion_crop.py static-ROI crop
PI_MIN_CROP = 160
SPECIES_IMG = 224
SPECIES_ALPHA = 1.0
SPECIES_CONTEXT = 1.5              # deployed rule: square crop, side = 1.5 x the box's long side
SPECIES_MIN_CROP = 48
TRAIN_CONTEXT_RANGE = (1.25, 2.2)  # bird fills 45-80% of the crop side
TRAIN_SHIFT = 0.08                 # crop-centre jitter, fraction of crop side
TRAIN_BOX_JITTER = 0.12            # hand-box jitter (scale and shift) to mimic detector error
LOOSE_BIRD_FRACTION = (0.12, 0.45) # fallback crops, as small as the bird in a static-ROI crop
CROP_MIX = {"detector": 0.45, "gt_box": 0.35, "loose": 0.20}
REGIONAL_EXTRA_SPECIES = [
    "American Goldfinch", "House Finch", "Dark-eyed Junco", "White-throated Sparrow", "Carolina Wren",
    "European Starling", "Common Grackle", "Red-winged Blackbird", "Hairy Woodpecker", "Red-breasted Nuthatch",
    "Brown-headed Cowbird", "American Robin", "Northern Mockingbird", "Chipping Sparrow", "Purple Finch",
    "American Tree Sparrow", "Northern Flicker", "Pine Siskin", "Rock Pigeon", "Gray Catbird",
]
SPECIES_GROUPS = {
    "sparrow-like": ["House Sparrow", "Song Sparrow", "White-throated Sparrow", "Chipping Sparrow", "American Tree Sparrow",
                     "Dark-eyed Junco", "House Finch", "Purple Finch", "American Goldfinch", "Pine Siskin"],
    "dove/pigeon": ["Mourning Dove", "Rock Pigeon"],
    "jay/crow": ["Blue Jay", "American Crow"],
    "chickadee/titmouse": ["Black-capped Chickadee", "Tufted Titmouse"],
    "nuthatch": ["White-breasted Nuthatch", "Red-breasted Nuthatch"],
    "woodpecker": ["Downy Woodpecker", "Hairy Woodpecker", "Red-bellied Woodpecker", "Northern Flicker"],
    "cardinal": ["Northern Cardinal"],
    "blackbird-like": ["Common Grackle", "Red-winged Blackbird", "Brown-headed Cowbird", "European Starling"],
    "other songbird": ["Carolina Wren", "American Robin", "Northern Mockingbird", "Gray Catbird"],
}
SPECIES_ESP32_SHARE = 0.65
SPECIES_CLASS_BALANCE_POWER = 0.5  # per-class weight ~ (training visits) ** power
SPECIES_ARMS = ("esp32_plus_outside", "esp32_only")
OUTSIDE_INAT_MIN_SCORE = 0.90

# ---- Automation ----------------------------------------------------------------
AUTO_TARGET_PRECISION = 0.95       # auto-labelled validation visits must be at least this precise
MIN_TRAIN_VISITS_FOR_AUTO = 5      # species with fewer camera training visits always go to review
REVIEW_DISAGREEMENT_MIN_PROB = 0.5 # review_queue: model's other answer at least this confident

# ---- Training --------------------------------------------------------------------
BATCH_SIZE = 32
EXAMPLES_PER_EPOCH = 9600
HEAD_EPOCHS = 3
FINE_TUNE_EPOCHS = 30
MIN_EPOCHS = 12                    # no early stop before this many epochs in total
EARLY_STOP_PATIENCE = 8
HEAD_LR = 1e-3
FINE_TUNE_PEAK_LR = 2e-4
WEIGHT_DECAY = 1e-4
FINE_TUNE_FROM_BLOCK = "block_6_expand"
LABEL_SMOOTHING = 0.05
REPRESENTATIVE_SAMPLES = 300
TFLITE_THREADS = 4

if QUICK_RUN:
    EXAMPLES_PER_EPOCH, HEAD_EPOCHS, FINE_TUNE_EPOCHS, MIN_EPOCHS = 256, 1, 1, 1
    EARLY_STOP_PATIENCE, REPRESENTATIVE_SAMPLES = 1, 16


## 2. Environment and helpers


In [2]:
import hashlib, io, json, math, os, platform, random, re, time
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath, PureWindowsPath

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image, ImageOps

keras = tf.keras
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)


def portable_path(raw):
    """Windows path <-> WSL /mnt/<drive> path, whichever this kernel needs."""
    raw = str(raw).strip()
    win = PureWindowsPath(raw)
    if win.drive:
        if os.name == "nt":
            return Path(raw)
        return Path("/mnt") / win.drive.rstrip(":").lower() / Path(*win.parts[1:])
    m = re.match(r"^/mnt/([A-Za-z])/(.*)$", raw.replace("\\", "/"))
    if m and os.name == "nt":
        return Path(f"{m.group(1).upper()}:/") / Path(*PurePosixPath(m.group(2)).parts)
    return Path(raw)


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def json_safe(v):
    if isinstance(v, dict):
        return {str(k): json_safe(x) for k, x in v.items()}
    if isinstance(v, (list, tuple)):
        return [json_safe(x) for x in v]
    if isinstance(v, Path):
        return str(v)
    if isinstance(v, np.ndarray):
        return json_safe(v.tolist())
    if isinstance(v, np.generic):
        v = v.item()
    if isinstance(v, float) and not math.isfinite(v):
        return None
    return v


def write_json(path, data):
    Path(path).write_text(json.dumps(json_safe(data), indent=2, allow_nan=False), encoding="utf-8")


def first_existing(paths, kind="dir"):
    for p in paths:
        q = portable_path(p)
        if (q.is_dir() if kind == "dir" else q.is_file()):
            return q
    return None


DATASET_ROOT_PATH = portable_path(DATASET_ROOT)
created_utc = datetime.now(timezone.utc)
RUN_NAME = f"esp32_species_v4_1_{created_utc:%Y%m%dT%H%M%SZ}" + ("_quick" if QUICK_RUN else "")
RUN_DIR = portable_path(OUTPUT_ROOT) / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Python", platform.python_version(), "| TensorFlow", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("Run directory:", RUN_DIR)


I0000 00:00:1795103419.016190   16832 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795103419.067183   16832 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795103420.806800   16832 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Python 3.12.15 | TensorFlow 2.21.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Run directory: /mnt/c/Users/Joe/Desktop/ESP_Classifier/esp32_Classifier/Models/runs/esp32_species_v4_1_20261119T155021Z


## 3. Load annotations, inventory and capture sidecars

One record per image: boxes with species, plus camera ID, sensor, static ROI and deployment from the sidecar JSON. Images whose SHA-256 does not match the inventory are rejected.


In [3]:
ann = pd.read_csv(portable_path(ANNOTATIONS_CSV), encoding="utf-8-sig", low_memory=False)
inv = pd.read_csv(portable_path(INVENTORY_CSV), encoding="utf-8-sig", low_memory=False)
for col in ("xmin", "ymin", "xmax", "ymax", "width_px", "height_px"):
    ann[col] = pd.to_numeric(ann[col], errors="coerce")
assert inv["sample_id"].is_unique, "inventory sample_id values must be unique"
assert set(ann["sample_id"]) == set(inv["sample_id"]), "annotations and inventory list different images"


def parse_sidecar(path):
    d = json.loads(Path(path).read_text(encoding="utf-8"))
    cam = d.get("camera") or {}
    sensor = (cam.get("sensor") or {}).get("model") or d.get("camera_model") or "unknown"
    roi = ((cam.get("requested") or {}).get("motion") or {}).get("roi")
    ok_roi = (isinstance(roi, dict) and set(roi) >= {"x", "y", "width", "height"}
              and all(isinstance(roi[k], (int, float)) for k in ("x", "y", "width", "height"))
              and roi["width"] > 0 and roi["height"] > 0)
    camera_id = d.get("camera_id") or (d.get("device") or {}).get("camera_id")
    return {
        "camera_id": camera_id or "",
        "sensor": str(sensor),
        "roi": {k: float(roi[k]) for k in ("x", "y", "width", "height")} if ok_roi else None,
        "deployment_id": d.get("deployment_id") or "",
        "motion_selection_source": d.get("motion_selection_source") or "",
        "capture_source": d.get("source") or "",
        "boot_id": (d.get("device") or {}).get("boot_id") or "",
    }


t0 = time.perf_counter()
records, rejected = [], []
boxed_rows = ann[ann["review_state"].eq("boxed") & ann["xmin"].notna()]
boxes_by_sample = {sid: g for sid, g in boxed_rows.groupby("sample_id")}
for row in inv.itertuples(index=False):
    image_path = DATASET_ROOT_PATH / row.image_filename
    sidecar_path = DATASET_ROOT_PATH / row.sidecar_filename
    if not image_path.is_file() or not sidecar_path.is_file():
        rejected.append((row.sample_id, "missing_file")); continue
    data = image_path.read_bytes()
    if hashlib.sha256(data).hexdigest() != row.image_sha256:
        rejected.append((row.sample_id, "image_sha256_mismatch")); continue
    meta = parse_sidecar(sidecar_path)
    g = boxes_by_sample.get(row.sample_id)
    W, H = int(row.width_px), int(row.height_px)
    if g is not None:
        b = g[["xmin", "ymin", "xmax", "ymax"]].to_numpy(np.float32)
        b[:, [0, 2]] = b[:, [0, 2]].clip(0, W); b[:, [1, 3]] = b[:, [1, 3]].clip(0, H)
        keep = (b[:, 2] - b[:, 0] >= 2) & (b[:, 3] - b[:, 1] >= 2)
        boxes, species = b[keep], g["species_name"].fillna("").astype(str).to_numpy()[keep]
    else:
        boxes, species = np.zeros((0, 4), np.float32), np.array([], dtype=object)
    stamp = re.match(r"capture_(\d{8}T\d{6})_(\d{6})Z", row.sample_id)
    when = pd.Timestamp(datetime.strptime("".join(stamp.groups()), "%Y%m%dT%H%M%S%f"), tz="UTC")
    records.append({
        "sample_id": row.sample_id, "jpeg": data, "width": W, "height": H,
        "boxes": boxes, "species": species, "has_bird": row.review_state == "boxed",
        "time": when, **meta,
    })
if rejected:
    print(f"rejected {len(rejected)} images:", pd.Series([r for _, r in rejected]).value_counts().to_dict())
assert records, "no usable images"

frames = pd.DataFrame([{k: v for k, v in r.items() if k not in ("jpeg", "boxes", "species")} for r in records])
frames["n_boxes"] = [len(r["boxes"]) for r in records]
frames["date"] = frames["time"].dt.strftime("%Y-%m-%d")
frames["local_hour"] = frames["time"].dt.tz_convert("America/New_York").dt.hour
print(f"loaded {len(records):,} images ({sum(len(r['jpeg']) for r in records) / 1e6:.0f} MB) in {time.perf_counter() - t0:.0f}s")
display(pd.crosstab([frames["sensor"], frames["camera_id"].replace("", "legacy (no id)")], frames["has_bird"], margins=True))
display(frames["motion_selection_source"].replace("", "(not recorded)").value_counts().to_frame("captures"))


loaded 6,184 images (284 MB) in 184s


has_bird                      False  True   All
sensor camera_id                               
OV2640 esp32cam-389e8af4e9d4    185   429   614
       legacy (no id)           410   596  1006
OV5640 esp32cam-b8aa6d0d5020   1578  2957  4535
       esp32cam-f4ae6d0d5020     20     9    29
All                            2193  3991  6184

,captures
motion_selection_source,
(not recorded),4515
static_roi_fallback,1639
dynamic_motion_blob,30


## 4. Visits and the visit-level split

Visits come from the hub's own `server/visits.py` rule, so their IDs match the hub's `visits.jsonl`. A visit is a run of captures from one camera with less than `VISIT_GAP_SECONDS` between consecutive frames; a change of deployment or camera reboot always starts a new one. Its ID is the camera plus its first capture. Every labelled box becomes a *track* (visit, species). Whole visits go to validation and test, assigned per species (by the visit's majority species), so even rare species get some held-out visits when they have at least 3.

Visits from the same day share light and angle, so this test is kinder than v4's held-out-scene test. It answers "will it label tomorrow's birds at this feeder", not "will it work on a new camera angle".


In [4]:
import sys
sys.path.insert(0, str(portable_path(SERVER_DIR)))
from visits import CaptureRef, assign_visits  # the hub's own visit rule, so IDs match visits.jsonl

frames["camera_key"] = frames["camera_id"].replace("", "legacy")
visit_of_capture = assign_visits(
    [CaptureRef(sid, cam or None, dep or None, boot or None)
     for sid, cam, dep, boot in zip(frames["sample_id"], frames["camera_id"], frames["deployment_id"], frames["boot_id"])],
    gap_seconds=VISIT_GAP_SECONDS)
frames["visit_id"] = frames["sample_id"].map(visit_of_capture)
visit_of = dict(zip(frames["sample_id"], frames["visit_id"]))

boxes = pd.DataFrame([(i, k, sp, visit_of[r["sample_id"]]) for i, r in enumerate(records)
                      for k, sp in enumerate(r["species"]) if sp], columns=["index", "box", "species", "visit_id"])
visit_species = boxes.groupby("visit_id")["species"].agg(lambda s: s.value_counts().index[0])

rng_split = np.random.default_rng(SEED)
visit_split = {}
for species, visits_of_species in visit_species.groupby(visit_species):
    ids = list(visits_of_species.index)
    rng_split.shuffle(ids)
    n = len(ids)
    n_test = int(round(n * TEST_VISIT_FRACTION)) if n >= 3 else 0
    n_val = int(round(n * VAL_VISIT_FRACTION)) if n >= 3 else 0
    n_test, n_val = max(n_test, 1 if n >= 3 else 0), max(n_val, 1 if n >= 3 else 0)
    for k, v in enumerate(ids):
        visit_split[v] = "test" if k < n_test else "validation" if k < n_test + n_val else "train"
frames["split"] = frames["visit_id"].map(visit_split).fillna("train")
boxes["split"] = boxes["visit_id"].map(visit_split).fillna("train")

visits = frames.groupby("visit_id").agg(
    camera=("camera_key", "first"), sensor=("sensor", "first"), scene_roi=("roi", lambda r: json.dumps(r.iloc[0])),
    start=("time", "min"), end=("time", "max"), frames=("sample_id", "size"),
    bird_frames=("has_bird", "sum"), split=("split", "first"), first_capture=("sample_id", "first"))
visits["duration_s"] = (visits["end"] - visits["start"]).dt.total_seconds()
visits["species"] = visit_species.reindex(visits.index).fillna("")
visits["all_species"] = boxes.groupby("visit_id")["species"].agg(lambda s: "|".join(sorted(set(s)))).reindex(visits.index).fillna("")
visits.to_csv(RUN_DIR / "visits.csv")

summary_table = boxes.groupby("species").agg(boxes=("box", "size"), visits=("visit_id", "nunique"))
for split in ("train", "validation", "test"):
    summary_table[f"{split}_visits"] = boxes[boxes["split"].eq(split)].groupby("species")["visit_id"].nunique()
display(summary_table.fillna(0).astype(int).sort_values("boxes", ascending=False))
print(f"{len(visits):,} visits ({int((visits['bird_frames'] > 0).sum()):,} with birds) from {len(frames):,} captures")


,boxes,visits,train_visits,validation_visits,test_visits
species,,,,,
House Sparrow,4529,64,40,11,13
Mourning Dove,984,11,8,2,1
Blue Jay,313,19,12,4,3
American Crow,230,3,1,1,1
Northern Cardinal,194,15,11,3,1
Black-capped Chickadee,183,8,6,2,0
White-breasted Nuthatch,183,10,6,2,2
Tufted Titmouse,80,3,1,1,1
Song Sparrow,67,6,4,1,1


153 visits (93 with birds) from 6,184 captures


## 5. Crop geometry

`species_crop_bounds` is the deployed rule: a square crop whose side is `SPECIES_CONTEXT` × the box's long side, centred on the box and shifted inside the frame. Copy this function into the server (see `DEPLOY.md`).


In [5]:
def normalized_roi_bounds(roi, width, height):
    x, y, w, h = (float(roi[k]) for k in ("x", "y", "width", "height"))
    return (max(0, int(math.floor(x * width))), max(0, int(math.floor(y * height))),
            min(width, int(math.ceil((x + w) * width))), min(height, int(math.ceil((y + h) * height))))


def final_crop_bounds(bounds, frame_w, frame_h, margin_fraction=PI_MARGIN_FRACTION, minimum_size=PI_MIN_CROP):
    left, top, right, bottom = bounds
    width, height = right - left, bottom - top
    tw = min(frame_w, max(minimum_size, int(math.ceil(width * (1 + margin_fraction)))))
    th = min(frame_h, max(minimum_size, int(math.ceil(height * (1 + margin_fraction)))))
    cx, cy = (left + right) / 2, (top + bottom) / 2
    cl = min(max(0, int(math.floor(cx - tw / 2))), frame_w - tw)
    ct = min(max(0, int(math.floor(cy - th / 2))), frame_h - th)
    return cl, ct, cl + tw, ct + th


def clip_rect(l, t, r, b, W, H, min_size=8):
    w, h = min(max(r - l, min_size), W), min(max(b - t, min_size), H)
    l = min(max(l, 0), W - w); t = min(max(t, 0), H - h)
    return int(math.floor(l)), int(math.floor(t)), int(math.floor(l)) + int(round(w)), int(math.floor(t)) + int(round(h))


def species_crop_bounds(box, frame_w, frame_h, context=SPECIES_CONTEXT, min_size=SPECIES_MIN_CROP, shift=(0.0, 0.0)):
    """DEPLOYED RULE. box = (left, top, right, bottom) in frame pixels -> half-open integer crop inside the frame."""
    left, top, right, bottom = (float(v) for v in box)
    side = max(right - left, bottom - top) * context
    side = max(side, float(min_size))
    cx = (left + right) / 2 + shift[0] * side
    cy = (top + bottom) / 2 + shift[1] * side
    return clip_rect(cx - side / 2, cy - side / 2, cx + side / 2, cy + side / 2, frame_w, frame_h, min_size)


def visible_fractions(boxes_px, crop):
    if not len(boxes_px):
        return np.zeros(0, np.float32)
    l, t, r, b = crop
    iw = np.clip(np.minimum(boxes_px[:, 2], r) - np.maximum(boxes_px[:, 0], l), 0, None)
    ih = np.clip(np.minimum(boxes_px[:, 3], b) - np.maximum(boxes_px[:, 1], t), 0, None)
    area = np.maximum((boxes_px[:, 2] - boxes_px[:, 0]) * (boxes_px[:, 3] - boxes_px[:, 1]), 1e-6)
    return (iw * ih / area).astype(np.float32)


def iou(a, b):
    iw = max(0.0, min(a[2], b[2]) - max(a[0], b[0])); ih = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    return inter / max((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter, 1e-6)


def pi_crop(record):
    """The crop the Pi hands the detector: static ROI (+20%, >=160 px), or the whole frame without a usable ROI."""
    roi = record["roi"]
    if roi is None or roi["width"] >= 0.95:
        return 0, 0, record["width"], record["height"]
    bounds = normalized_roi_bounds(roi, record["width"], record["height"])
    return final_crop_bounds(bounds, record["width"], record["height"])


## 6. Localise the birds: run both detectors on every capture

Two localisers are run on every capture, each on the crop it would see on the hub:

- **`pi_detector`** is the hub's current detector. It runs through the server's own `bird_inference.py` code, on the Pi-selected crop (static ROI with margin).
- **`esp32_roi_v3`** is the v3 ESP32 ROI detector (128 px, int8). It runs on a square around the same crop.

For each one, every labelled box is matched to the detection it overlaps most (IoU ≥ `DETECTOR_MATCH_IOU`). With `LOCALIZER = "auto"`, whichever finds more labelled birds supplies the boxes for the species crops. The table it prints is the answer to "which detector should the hub use to centre the species crop". Results are cached per model hash.


In [6]:
import shutil, sys, types
sys.path.insert(0, str(portable_path(SERVER_DIR)))
import bird_inference as hub  # the Pi's inference module

try:
    from ai_edge_litert.interpreter import Interpreter as _TFLite
except ImportError:
    _TFLite = tf.lite.Interpreter  # the hub's own lookup misses tf.lite in TF >= 2.20


def load_pi_detector():
    path = portable_path(DETECTOR_MODEL)
    try:
        contract = hub.DetectorContract.beside(path)
    except ValueError as error:
        # A Windows git checkout turns the JSON sidecars into CRLF, so their SHA-256 no longer matches the
        # contract. Use LF-normalised copies (the bytes the Pi has); the contract still verifies them.
        if "SHA-256" not in str(error):
            raise
        local = RUN_DIR / "_pi_detector"; local.mkdir(exist_ok=True)
        for f in path.parent.iterdir():
            if f.suffix == ".json":
                (local / f.name).write_bytes(f.read_bytes().replace(b"\r\n", b"\n"))
            elif f == path:
                shutil.copyfile(f, local / f.name)
        path = local / path.name
        contract = hub.DetectorContract.beside(path)
        print("pi_detector JSON files had CRLF line endings; using LF-normalised copies (hashes verified)")
    model = hub.DetectorModel(path, image_size=contract.image_size, num_threads=TFLITE_THREADS, interpreter_factory=_TFLite)
    contract.validate_model(model)
    box_pos, score_pos = int(contract.box_spec["output_position"]), int(contract.score_spec["output_position"])

    def run(rgb, record):
        crop = pi_crop(record)
        l, t, r, b = crop
        out = model.predict(hub.letterbox_array(rgb[t:b, l:r], contract.image_size, padding_value=contract.padding_value))
        return hub.decode_detector_outputs(out[box_pos], out[score_pos], contract, crop)
    return run, sha256_file(path)[:12]


def square_around(crop, W, H):
    l, t, r, b = crop
    side = min(max(r - l, b - t), W, H)
    cx, cy = (l + r) / 2, (t + b) / 2
    L = int(min(max(0, math.floor(cx - side / 2)), W - side)); T = int(min(max(0, math.floor(cy - side / 2)), H - side))
    return L, T, L + side, T + side


def load_roi_v3():
    folder = portable_path(ROI_V3_DIR)
    contract = json.loads((folder / "esp32_detector_contract.json").read_text(encoding="utf-8"))
    path = folder / contract["model_file"]
    interpreter = _TFLite(model_path=str(path), num_threads=TFLITE_THREADS); interpreter.allocate_tensors()
    inp, outs = interpreter.get_input_details()[0], interpreter.get_output_details()
    size = int(contract["input"]["shape"][1])
    anchors = np.asarray(contract["anchors_normalized_wh"], np.float32)
    score_detail = next(o for o in outs if o["shape"][-1] == len(anchors))
    box_detail = next(o for o in outs if o["shape"][-1] == 4 * len(anchors))
    grid = int(box_detail["shape"][1])
    dec = contract["decode"]
    spec = types.SimpleNamespace(box_spec={"shape": [1, grid, grid, len(anchors), 4]}, score_spec={"shape": [1, grid, grid, len(anchors), 1]},
                                 anchors=anchors, candidate_threshold=float(dec["candidate_score_threshold"]), image_size=size,
                                 max_detections=int(dec["max_detections"]), nms_iou=float(dec["nms_iou"]))
    pad = int(contract.get("pi_crop", {}).get("pad_value", 114))

    def dequant(detail):
        raw = interpreter.get_tensor(detail["index"]).astype(np.float32)
        scale, zero = detail["quantization"]
        return (raw - zero) * scale if scale else raw

    def run(rgb, record):
        crop = square_around(pi_crop(record), record["width"], record["height"])
        l, t, r, b = crop
        pixels = hub.letterbox_array(rgb[t:b, l:r], size, padding_value=pad)
        x = np.clip(np.rint(pixels), 0, 255).astype(inp["dtype"])[None] if np.issubdtype(inp["dtype"], np.integer) else pixels[None]
        interpreter.set_tensor(inp["index"], x); interpreter.invoke()
        box = dequant(box_detail).reshape(spec.box_spec["shape"]); score = dequant(score_detail).reshape(spec.score_spec["shape"])
        return hub.decode_detector_outputs(box, score, spec, crop)
    return run, sha256_file(path)[:12]


localizers = {}
for name, loader in (("esp32_roi_v3", load_roi_v3), ("pi_detector", load_pi_detector)):
    try:
        localizers[name] = loader()
    except Exception as error:
        print(f"{name} unavailable:", repr(error))
assert localizers, "no localiser could be loaded"

cache_dir = portable_path(DETECTOR_CACHE_DIR); cache_dir.mkdir(parents=True, exist_ok=True)
detections_by = {}
for name, (run, sha) in localizers.items():
    cache_path = cache_dir / f"{name}_{sha}.json"
    cache = json.loads(cache_path.read_text()) if cache_path.is_file() else {}
    t0, ran, found = time.perf_counter(), 0, []
    for r in records:
        key = r["sample_id"] + "|" + json.dumps(r["roi"], sort_keys=True)
        if key not in cache:
            rgb = np.asarray(Image.open(io.BytesIO(r["jpeg"])).convert("RGB"))
            cache[key] = [[d["score"], d["box"]["left"], d["box"]["top"], d["box"]["right"], d["box"]["bottom"]] for d in run(rgb, r)]
            ran += 1
        found.append([d for d in cache[key] if d[0] >= DETECTOR_MIN_SCORE])
    if ran:
        cache_path.write_text(json.dumps(cache))
    detections_by[name] = found
    print(f"{name}: {ran:,} new captures scored in {time.perf_counter() - t0:.0f}s")


def match_boxes(found):
    matched = {}
    for row in boxes.rename(columns={"index": "record_idx"}).itertuples(index=False):
        gt = records[row.record_idx]["boxes"][row.box]
        best = max(((iou(gt, d[1:]), d) for d in found[row.record_idx]), default=(0.0, None), key=lambda x: x[0])
        if best[1] is not None and best[0] >= DETECTOR_MATCH_IOU:
            matched[(row.record_idx, row.box)] = (np.asarray(best[1][1:], np.float32), best[1][0], best[0])
    return matched


matches = {name: match_boxes(found) for name, found in detections_by.items()}
boxes["in_pi_crop"] = [visible_fractions(records[i]["boxes"][k:k + 1], pi_crop(records[i]))[0] >= 0.9
                       for i, k in zip(boxes["index"], boxes["box"])]
for name, m in matches.items():
    boxes[f"found_{name}"] = [(i, k) in m for i, k in zip(boxes["index"], boxes["box"])]
    boxes[f"iou_{name}"] = [m.get((i, k), (None, None, 0.0))[2] for i, k in zip(boxes["index"], boxes["box"])]
match_table = boxes.groupby("species")[["in_pi_crop"] + [f"found_{n}" for n in matches]].mean()
match_table.loc["ALL"] = boxes[["in_pi_crop"] + [f"found_{n}" for n in matches]].mean()
display(match_table.round(3))
match_table.to_csv(RUN_DIR / "localisation_by_species.csv")

chosen_localizer = max(matches, key=lambda n: len(matches[n])) if LOCALIZER == "auto" else LOCALIZER
det_box_of = matches[chosen_localizer]
det_sha = localizers[chosen_localizer][1]
boxes["detector_found"] = boxes[f"found_{chosen_localizer}"]
print(f"species crops use: {chosen_localizer} (finds {boxes['detector_found'].mean():.1%} of labelled birds at IoU >= {DETECTOR_MATCH_IOU})")


/home/joe/venvs/ESPclassifier/lib/python3.12/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


pi_detector JSON files had CRLF line endings; using LF-normalised copies (hashes verified)
esp32_roi_v3: 0 new captures scored in 0s
pi_detector: 0 new captures scored in 0s


,in_pi_crop,found_esp32_roi_v3,found_pi_detector
species,,,
American Crow,1.000,0.939,0.017
Black-capped Chickadee,0.989,0.656,0.158
Blue Jay,0.891,0.952,0.160
Downy Woodpecker,0.941,1.000,0.824
House Sparrow,0.960,0.740,0.223
Mourning Dove,0.814,0.971,0.756
Northern Cardinal,0.804,0.943,0.531
Red-bellied Woodpecker,1.000,1.000,0.600
Song Sparrow,1.000,0.657,0.000


species crops use: esp32_roi_v3 (finds 78.9% of labelled birds at IoU >= 0.3)


## 7. Species crops and evaluation tables

Training draws come in three kinds. **detector** is the matched detector box with context and shift jitter. **gt_box** is the hand box, jittered to look like detector error. **loose** puts the bird at 12–45% of the crop side, for when the hub has to fall back to the ROI. Any crop that takes in more than half of a bird of another species is redrawn.

The evaluation kinds are `chain` (the detector crop when the detector found the bird, otherwise the Pi's static-ROI crop; this is what the hub would really do), `detector`, `gt_box` and `static_roi`.


In [7]:
def intruder_free(record, k, crop):
    vis = visible_fractions(record["boxes"], crop)
    other = record["species"] != record["species"][k]
    return not np.any(other & (vis > 0.5))


def jitter_box(rng, box, amount):
    l, t, r, b = (float(v) for v in box)
    w, h = r - l, b - t
    s = math.exp(rng.uniform(-amount, amount))
    cx, cy = (l + r) / 2 + rng.normal(0, amount / 2) * w, (t + b) / 2 + rng.normal(0, amount / 2) * h
    return cx - w * s / 2, cy - h * s / 2, cx + w * s / 2, cy + h * s / 2


def loose_crop(rng, W, H, box):
    l, t, r, b = (float(v) for v in box)
    bw, bh = max(r - l, 2.0), max(b - t, 2.0)
    side = max(bw, bh) / rng.uniform(*LOOSE_BIRD_FRACTION)
    aspect = math.exp(rng.uniform(-0.4, 0.4))
    cw, ch = max(side * math.sqrt(aspect), bw * 1.05), max(side / math.sqrt(aspect), bh * 1.05)
    cx = (l + r) / 2 + rng.uniform(-1, 1) * (cw - bw) / 2 * 0.8
    cy = (t + b) / 2 + rng.uniform(-1, 1) * (ch - bh) / 2 * 0.8
    return clip_rect(cx - cw / 2, cy - ch / 2, cx + cw / 2, cy + ch / 2, W, H, 16)


def train_species_crop(rng, i, k, kind):
    r = records[i]
    W, H = r["width"], r["height"]
    for _ in range(6):
        if kind == "detector" and (i, k) in det_box_of:
            base = det_box_of[(i, k)][0]
        else:
            base = jitter_box(rng, r["boxes"][k], TRAIN_BOX_JITTER) if kind != "loose" else r["boxes"][k]
        if kind == "loose":
            crop = loose_crop(rng, W, H, base)
        else:
            crop = species_crop_bounds(base, W, H, context=rng.uniform(*TRAIN_CONTEXT_RANGE),
                                       shift=tuple(rng.uniform(-TRAIN_SHIFT, TRAIN_SHIFT, 2)))
        if intruder_free(r, k, crop):
            return crop
    return species_crop_bounds(r["boxes"][k], W, H)


def eval_rows(split):
    rows = []
    for row in boxes[boxes["split"].eq(split)].rename(columns={"index": "record_idx"}).itertuples(index=False):
        r = records[row.record_idx]
        W, H = r["width"], r["height"]
        crops = {"gt_box": species_crop_bounds(r["boxes"][row.box], W, H)}
        if (row.record_idx, row.box) in det_box_of:
            crops["detector"] = species_crop_bounds(det_box_of[(row.record_idx, row.box)][0], W, H)
        pc = pi_crop(r)
        if r["roi"] is not None and row.in_pi_crop and intruder_free(r, row.box, pc):
            crops["static_roi"] = pc
        chain = crops.get("detector", crops.get("static_roi"))
        if chain is not None:
            crops["chain"] = chain
        for kind, crop in crops.items():
            rows.append({"index": row.record_idx, "box": row.box, "sample_id": r["sample_id"], "visit_id": row.visit_id,
                         "species": row.species, "kind": kind,
                         "left": crop[0], "top": crop[1], "right": crop[2], "bottom": crop[3]})
    return pd.DataFrame(rows)


species_val = eval_rows("validation")
species_test = eval_rows("test")
display(pd.crosstab(species_test["species"], species_test["kind"]))
train_boxes = boxes[boxes["split"].eq("train")]
camera_train_species = sorted(train_boxes["species"].unique())
train_visits_per_species = train_boxes.groupby("species")["visit_id"].nunique()


kind,chain,detector,gt_box,static_roi
species,,,,
American Crow,78,78,80,0
Blue Jay,26,26,26,21
House Sparrow,277,237,286,270
Mourning Dove,13,13,13,0
Northern Cardinal,2,2,2,2
Song Sparrow,4,4,4,0
Tufted Titmouse,0,0,1,0
White-breasted Nuthatch,8,8,8,6


## 8. Outside crops (iNaturalist + CUB), training only

Only species in the label set are loaded: the ones seen on camera plus `REGIONAL_EXTRA_SPECIES`. Images are EXIF-rotated only when that matches the manifest's recorded size, so the boxes stay aligned. If no image root is found, this section reports it and the species model trains on ESP32 crops only.


In [8]:
def norm_name(name):
    name = re.sub(r"[-_']", " ", str(name).lower())
    return re.sub(r"\s+", " ", name).strip()


CUB_ALIASES = {"cardinal": "northern cardinal", "mockingbird": "northern mockingbird", "tree sparrow": "american tree sparrow"}
camera_species = sorted(boxes["species"].unique())
wanted = {norm_name(s): s for s in camera_species + REGIONAL_EXTRA_SPECIES}
outside = []  # dicts: jpeg, width, height, box, species, source
outside_report = {}


def load_outside_image(path, expected_w, expected_h, box):
    with Image.open(path) as im:
        raw_size = im.size
        rotated = ImageOps.exif_transpose(im)
        if (int(expected_w), int(expected_h)) == rotated.size:
            image = rotated.convert("RGB")
        elif (int(expected_w), int(expected_h)) == raw_size:
            image = im.convert("RGB")
        else:
            return None
    scale = min(1.0, 448 / max(image.size))
    if scale < 1.0:
        image = image.resize((max(1, round(image.width * scale)), max(1, round(image.height * scale))), Image.BILINEAR)
    buffer = io.BytesIO(); image.save(buffer, format="JPEG", quality=92)
    b = np.asarray(box, np.float32) * scale
    b[[0, 2]] = b[[0, 2]].clip(0, image.width); b[[1, 3]] = b[[1, 3]].clip(0, image.height)
    if b[2] - b[0] < 8 or b[3] - b[1] < 8:
        return None
    return buffer.getvalue(), image.width, image.height, b


manifest_dir = next((portable_path(d) for d in OUTSIDE_MANIFEST_DIRS
                     if (portable_path(d) / "bird_manifest.csv").is_file()), None)
inat_root = first_existing(INAT_IMAGE_ROOTS)
cub_root = first_existing(CUB_IMAGE_ROOTS)
print("manifests:", manifest_dir, "| iNat root:", inat_root, "| CUB root:", cub_root)
t0 = time.perf_counter()

if manifest_dir is not None:
    inat = pd.read_csv(manifest_dir / "bird_manifest.csv", encoding="utf-8-sig", low_memory=False,
                       usecols=["image_id", "relative_path", "absolute_path", "width", "height", "common_name"])
    inat["key"] = inat["common_name"].map(norm_name)
    inat = inat[inat["key"].isin(wanted)]
    inat_boxes = pd.read_csv(manifest_dir / "inaturalist_bird_boxes.csv", encoding="utf-8-sig")
    inat_boxes["image_id"] = inat_boxes["source_id"].str.split(":").str[1].astype(int)
    inat_boxes = inat_boxes[inat_boxes["score"] >= OUTSIDE_INAT_MIN_SCORE].sort_values("score", ascending=False).drop_duplicates("image_id")
    inat = inat.merge(inat_boxes[["image_id", "x", "y", "w", "h"]], on="image_id")
    if QUICK_RUN:
        inat = inat.groupby("key").head(3)
    loaded = 0
    for row in inat.itertuples(index=False):
        rel = PurePosixPath(str(row.relative_path).replace("\\", "/"))
        candidates = [portable_path(row.absolute_path), inat_root.joinpath(*rel.parts) if inat_root else None,
                      inat_root.joinpath(*rel.parts[1:]) if inat_root and len(rel.parts) > 1 else None]
        path = next((p for p in candidates if p is not None and p.is_file()), None)
        if path is None:
            continue
        result = load_outside_image(path, row.width, row.height, (row.x, row.y, row.x + row.w, row.y + row.h))
        if result:
            jpeg, w, h, b = result
            outside.append({"jpeg": jpeg, "width": w, "height": h, "box": b, "species": wanted[row.key], "source": "inaturalist"})
            loaded += 1
    outside_report["inaturalist"] = {"candidates": len(inat), "loaded": loaded}

if manifest_dir is not None and (manifest_dir / "cub_200_2011_image_manifest.csv").is_file():
    cub = pd.read_csv(manifest_dir / "cub_200_2011_image_manifest.csv", low_memory=False,
                      usecols=["relative_path", "image_path", "class_name_friendly", "bbox_x", "bbox_y", "bbox_width", "bbox_height",
                               "width_px", "height_px"])
    cub["key"] = cub["class_name_friendly"].map(norm_name).map(lambda k: CUB_ALIASES.get(k, k))
    cub = cub[cub["key"].isin(wanted)]
    if QUICK_RUN:
        cub = cub.groupby("key").head(3)
    loaded = 0
    for row in cub.itertuples(index=False):
        options = [portable_path(row.image_path), cub_root.joinpath(*PurePosixPath(str(row.relative_path).replace("\\", "/")).parts) if cub_root else None]
        path = next((p for p in options if p is not None and p.is_file()), None)
        if path is None:
            continue
        result = load_outside_image(path, row.width_px, row.height_px,
                                    (row.bbox_x, row.bbox_y, row.bbox_x + row.bbox_width, row.bbox_y + row.bbox_height))
        if result:
            jpeg, w, h, b = result
            outside.append({"jpeg": jpeg, "width": w, "height": h, "box": b, "species": wanted[row.key], "source": "cub"})
            loaded += 1
    outside_report["cub"] = {"candidates": len(cub), "loaded": loaded}

print(f"outside crops loaded in {time.perf_counter() - t0:.0f}s:", outside_report)
outside_counts = pd.Series([o["species"] for o in outside], dtype=object).value_counts()
display(outside_counts.to_frame("outside images"))


manifests: /mnt/d/Datasets/ESP32Cam_confirmed/Manifests | iNat root: /mnt/d/Datasets/iNaturalist | CUB root: /mnt/d/Datasets/CUB_200/CUB_200_2011/images
outside crops loaded in 63s: {'inaturalist': {'candidates': 1256, 'loaded': 1256}, 'cub': {'candidates': 1136, 'loaded': 1136}}


,outside images
American Tree Sparrow,107
White-throated Sparrow,107
Purple Finch,106
Chipping Sparrow,105
Northern Mockingbird,105
Carolina Wren,102
Gray Catbird,100
American Goldfinch,100
American Crow,99
Song Sparrow,98


## 9. TensorFlow pipeline, model builder and training loop

The same as v4, except that early stopping waits for `MIN_EPOCHS`.


In [9]:
AUTOTUNE = tf.data.AUTOTUNE


def decode_crop_letterbox(jpeg, crop, size):
    """crop = [left, top, right, bottom] int32, half-open, inside the frame."""
    window = tf.stack([crop[1], crop[0], crop[3] - crop[1], crop[2] - crop[0]])
    image = tf.io.decode_and_crop_jpeg(jpeg, window, channels=3)
    image = tf.image.resize_with_pad(tf.cast(image, tf.float32), size, size, antialias=True)
    return tf.ensure_shape(image, (size, size, 3))


def camera_augment(image, strong_greyscale=0.05):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, 18.0)
    image = tf.image.random_contrast(image, 0.85, 1.15)
    image = tf.image.random_saturation(image, 0.8, 1.2)
    image = image * tf.random.uniform([1, 1, 3], 0.92, 1.08)  # white-balance drift between cameras
    grey = tf.image.grayscale_to_rgb(tf.image.rgb_to_grayscale(image))
    image = tf.where(tf.random.uniform([]) < strong_greyscale, grey, image)
    return tf.clip_by_value(image, 0.0, 255.0)


def esp32_look(image, size):
    """Make a sharp iNat/CUB crop look like an ESP32 crop: low native resolution, JPEG, noise, colour cast."""
    native = tf.random.uniform([], 56, 170, dtype=tf.int32)
    image = tf.image.resize(tf.image.resize(image, [native, native], antialias=True), [size, size])
    quality = tf.random.uniform([], 15, 60, dtype=tf.int32)
    image = tf.cast(tf.image.adjust_jpeg_quality(tf.cast(tf.clip_by_value(image, 0, 255), tf.uint8), quality), tf.float32)
    image = image + tf.random.normal(tf.shape(image), 0.0, tf.random.uniform([], 1.0, 8.0))
    image = image * tf.random.uniform([1, 1, 3], 0.8, 1.2)
    return tf.clip_by_value(image, 0.0, 255.0)


def make_train_ds(generator, size, n_classes, onehot):
    signature = (tf.TensorSpec((), tf.string), tf.TensorSpec((4,), tf.int32),
                 tf.TensorSpec((), tf.int32), tf.TensorSpec((), tf.bool))

    def prepare(jpeg, crop, label, outside):
        image = decode_crop_letterbox(jpeg, crop, size)
        image = tf.cond(outside, lambda: esp32_look(image, size), lambda: image)
        image = camera_augment(image)
        target = tf.one_hot(label, n_classes) if onehot else tf.cast(tf.reshape(label, (1,)), tf.float32)
        return image, target

    return (tf.data.Dataset.from_generator(generator, output_signature=signature)
            .map(prepare, num_parallel_calls=AUTOTUNE)
            .batch(BATCH_SIZE, drop_remainder=True).prefetch(AUTOTUNE))


def make_eval_ds(jpegs, crops, size):
    ds = tf.data.Dataset.from_tensor_slices((list(jpegs), np.asarray(crops, np.int32)))
    return ds.map(lambda j, c: decode_crop_letterbox(j, c, size), num_parallel_calls=AUTOTUNE).batch(64).prefetch(AUTOTUNE)


def build_classifier(size, alpha, n_out, activation, name):
    backbone = keras.applications.MobileNetV2(input_shape=(size, size, 3), alpha=alpha, include_top=False, weights=WEIGHTS)
    backbone.trainable = False
    inputs = keras.Input((size, size, 3), name="image")
    x = keras.layers.Rescaling(1.0 / 127.5, offset=-1.0, name="scale_to_mobilenet")(inputs)
    x = backbone(x, training=False)  # BatchNorm stays in inference mode, even while fine-tuning
    x = keras.layers.GlobalAveragePooling2D(name="global_average_pool")(x)
    x = keras.layers.Dropout(0.3, name="dropout")(x)
    outputs = keras.layers.Dense(n_out, activation=activation, name="output")(x)
    return keras.Model(inputs, outputs, name=name), backbone


def unfreeze_from(backbone, layer_name):
    backbone.trainable = True
    names = [layer.name for layer in backbone.layers]
    start = names.index(layer_name)
    for k, layer in enumerate(backbone.layers):
        layer.trainable = k >= start and not isinstance(layer, keras.layers.BatchNormalization)
    return sum(1 for layer in backbone.layers if layer.trainable)


class SelectBest(keras.callbacks.Callback):
    """Score the validation crops each epoch with `score_fn`; keep the best weights; stop when stale."""

    def __init__(self, score_fn, path, patience, log):
        super().__init__()
        self.score_fn, self.path, self.patience, self.log = score_fn, str(path), patience, log
        self.best, self.wait = -np.inf, 0

    def on_epoch_end(self, epoch, logs=None):
        score = float(self.score_fn(self.model))
        self.log.append({"epoch": len(self.log) + 1, "loss": float((logs or {}).get("loss", np.nan)), "val_score": score})
        print(f"  validation score {score:.4f} (best {max(self.best, score):.4f}; higher is better)")
        if score > self.best:
            self.best, self.wait = score, 0
            self.model.save_weights(self.path)
        else:
            self.wait += 1
            if self.wait >= self.patience and len(self.log) >= MIN_EPOCHS:
                self.model.stop_training = True


def train_two_stage(model, backbone, train_ds, loss, score_fn, out_dir):
    history = []
    steps = max(1, EXAMPLES_PER_EPOCH // BATCH_SIZE)
    best_path = Path(out_dir) / "best.weights.h5"
    model.compile(optimizer=keras.optimizers.Adam(HEAD_LR), loss=loss)
    print("stage 1: head only")
    selector = SelectBest(score_fn, best_path, EARLY_STOP_PATIENCE, history)
    model.fit(train_ds, epochs=HEAD_EPOCHS, steps_per_epoch=steps, callbacks=[selector], verbose=2)
    model.load_weights(best_path)
    trainable = unfreeze_from(backbone, FINE_TUNE_FROM_BLOCK)
    print(f"stage 2: fine-tune {trainable} backbone layers from {FINE_TUNE_FROM_BLOCK}")
    total = steps * FINE_TUNE_EPOCHS
    schedule = keras.optimizers.schedules.CosineDecay(
        FINE_TUNE_PEAK_LR * 0.05, decay_steps=max(1, total - steps), alpha=0.02,
        warmup_target=FINE_TUNE_PEAK_LR, warmup_steps=steps)
    model.compile(optimizer=keras.optimizers.AdamW(schedule, weight_decay=WEIGHT_DECAY), loss=loss)
    selector.wait = 0
    model.fit(train_ds, epochs=FINE_TUNE_EPOCHS, steps_per_epoch=steps, callbacks=[selector], verbose=2)
    model.load_weights(best_path)
    pd.DataFrame(history).to_csv(Path(out_dir) / "training_history.csv", index=False)
    return history


try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    Interpreter = tf.lite.Interpreter


def export_tflite(model, representative_images, out_dir, stem="model"):
    out_dir = Path(out_dir)
    float_path, uint8_path = out_dir / f"{stem}_float32.tflite", out_dir / f"{stem}_uint8.tflite"
    float_path.write_bytes(tf.lite.TFLiteConverter.from_keras_model(model).convert())
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = lambda: ([img[None].astype(np.float32)] for img in representative_images)
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.uint8
    converter.inference_output_type = tf.uint8
    uint8_path.write_bytes(converter.convert())
    model.save(out_dir / f"{stem}.keras")
    return float_path, uint8_path


def tflite_run(path, images_ds):
    """-> raw outputs (N, K), dequantized outputs (N, K), tensor info."""
    interpreter = Interpreter(model_path=str(path), num_threads=TFLITE_THREADS)
    interpreter.allocate_tensors()
    inp, out = interpreter.get_input_details()[0], interpreter.get_output_details()[0]
    raw_rows = []
    for batch in images_ds:
        for image in batch.numpy():
            x = image[None].astype(np.float32)
            if np.issubdtype(inp["dtype"], np.integer):
                scale, zero = inp["quantization"]
                x = np.clip(np.rint(x / scale + zero), 0, 255).astype(inp["dtype"])
            interpreter.set_tensor(inp["index"], x)
            interpreter.invoke()
            raw_rows.append(interpreter.get_tensor(out["index"]).reshape(-1).copy())
    raw = np.stack(raw_rows)
    if np.issubdtype(out["dtype"], np.integer):
        scale, zero = out["quantization"]
        values = (raw.astype(np.float32) - zero) * scale
    else:
        values = raw.astype(np.float32)
    info = {"input_shape": [int(v) for v in inp["shape"]], "input_dtype": np.dtype(inp["dtype"]).name,
            "input_quantization": [float(inp["quantization"][0]), int(inp["quantization"][1])],
            "output_dtype": np.dtype(out["dtype"]).name,
            "output_quantization": [float(out["quantization"][0]), int(out["quantization"][1])]}
    return raw, values, info


## 10. Train the species arms

Each training draw picks a species (weighted by its training *visits* to the power `SPECIES_CLASS_BALANCE_POWER`), then a visit, then a frame and box from that visit. Every visit therefore counts the same however many frames it has. The checkpoint and the arm are chosen on **class-balanced validation log-loss** over the camera species, with probabilities renormalised over those species, so arms with different label sets compare fairly.


In [10]:
def arm_classes(arm):
    if arm == "esp32_only" or not outside:
        return list(camera_train_species)
    return sorted(set(camera_train_species) | set(outside_counts.index))


val_scored = species_val[species_val["kind"].isin(["detector", "gt_box"]) & species_val["species"].isin(camera_train_species)].reset_index(drop=True)
val_scored_ds = make_eval_ds([records[i]["jpeg"] for i in val_scored["index"]],
                             val_scored[["left", "top", "right", "bottom"]].to_numpy(np.int32), SPECIES_IMG)


def balanced_nll(probs, classes, table):
    """Mean over camera species of the mean -log p(true), with p renormalised over camera species."""
    cols = [classes.index(c) for c in camera_train_species]
    p = probs[:, cols]
    p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-9, None)
    truth = table["species"].map({c: k for k, c in enumerate(camera_train_species)}).to_numpy()
    nll = -np.log(np.clip(p[np.arange(len(p)), truth], 1e-6, 1.0))
    return float(np.mean([nll[truth == c].mean() for c in np.unique(truth)]))


def make_score(classes):
    return lambda model: -balanced_nll(model.predict(val_scored_ds, verbose=0), classes, val_scored)


def species_generator_factory(classes, use_outside):
    index_of = {c: k for k, c in enumerate(classes)}
    by_class = {}
    for c, g in train_boxes.groupby("species"):
        if c in index_of:
            by_class[c] = [v[["index", "box"]].to_numpy() for _, v in g.groupby("visit_id")]
    esp_classes = sorted(by_class)
    esp_w = np.array([len(by_class[c]) ** SPECIES_CLASS_BALANCE_POWER for c in esp_classes], np.float64)
    esp_w /= esp_w.sum()
    out_by_class = {}
    if use_outside:
        for k, o in enumerate(outside):
            if o["species"] in index_of:
                out_by_class.setdefault(o["species"], []).append(k)
    out_classes = sorted(out_by_class)
    out_w = np.array([len(out_by_class[c]) ** SPECIES_CLASS_BALANCE_POWER for c in out_classes], np.float64) if out_classes else None
    if out_w is not None:
        out_w /= out_w.sum()
    esp_share = SPECIES_ESP32_SHARE if out_classes else 1.0
    kinds, kind_w = list(CROP_MIX), np.array(list(CROP_MIX.values()), np.float64) / sum(CROP_MIX.values())

    def generator():
        rng = np.random.default_rng(SEED + 31)
        while True:
            if rng.random() < esp_share:
                c = esp_classes[rng.choice(len(esp_classes), p=esp_w)]
                visit = by_class[c][rng.integers(len(by_class[c]))]
                i, k = visit[rng.integers(len(visit))]
                kind = kinds[rng.choice(len(kinds), p=kind_w)]
                yield records[i]["jpeg"], np.asarray(train_species_crop(rng, int(i), int(k), kind), np.int32), np.int32(index_of[c]), False
            else:
                c = out_classes[rng.choice(len(out_classes), p=out_w)]
                o = outside[out_by_class[c][rng.integers(len(out_by_class[c]))]]
                if rng.random() < 0.8:
                    crop = species_crop_bounds(jitter_box(rng, o["box"], TRAIN_BOX_JITTER), o["width"], o["height"],
                                               context=rng.uniform(*TRAIN_CONTEXT_RANGE))
                else:
                    crop = loose_crop(rng, o["width"], o["height"], o["box"])
                yield o["jpeg"], np.asarray(crop, np.int32), np.int32(index_of[c]), True
    return generator


arm_results = {}
for arm in [a for a in SPECIES_ARMS if not (a == "esp32_plus_outside" and not outside)]:
    arm_dir = RUN_DIR / arm
    arm_dir.mkdir(exist_ok=True)
    classes = arm_classes(arm)
    print(f"\n=== species arm {arm}: {len(classes)} classes ===")
    gen = species_generator_factory(classes, use_outside=(arm != "esp32_only"))
    ds = make_train_ds(gen, SPECIES_IMG, len(classes), onehot=True)
    model, backbone = build_classifier(SPECIES_IMG, SPECIES_ALPHA, len(classes), "softmax", "species_mobilenetv2")
    t0 = time.perf_counter()
    history = train_two_stage(model, backbone, ds, keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
                              make_score(classes), arm_dir)
    (arm_dir / "labels.txt").write_text("\n".join(classes) + "\n", encoding="utf-8")
    rep = []
    for jpeg, crop, _, is_out in gen():
        image = decode_crop_letterbox(jpeg, tf.constant(crop), SPECIES_IMG)
        rep.append((esp32_look(image, SPECIES_IMG) if is_out else image).numpy())
        if len(rep) >= REPRESENTATIVE_SAMPLES:
            break
    float_path, uint8_path = export_tflite(model, rep, arm_dir)
    arm_results[arm] = {"classes": classes, "dir": arm_dir, "float": float_path, "uint8": uint8_path,
                        "best_val_balanced_nll": -max(h["val_score"] for h in history),
                        "epochs": len(history), "minutes": (time.perf_counter() - t0) / 60}
    print(f"{arm}: best validation balanced log-loss {arm_results[arm]['best_val_balanced_nll']:.4f} "
          f"after {len(history)} epochs, {arm_results[arm]['minutes']:.1f} min")
    del model, backbone, ds
    keras.backend.clear_session()

best_arm = min(arm_results, key=lambda a: arm_results[a]["best_val_balanced_nll"])
print("lowest validation log-loss:", best_arm, "(final choice is made on automation in section 11)")


I0000 00:00:1795103672.206790   16832 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3582 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6



=== species arm esp32_plus_outside: 31 classes ===
stage 1: head only
Epoch 1/3


/home/joe/venvs/ESPclassifier/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1795103674.855249   16832 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1795103678.395536   16950 service.cc:153] XLA service 0x71f148035a60 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1795103678.395573   16950 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3060 Laptop GPU, Compute Capability 8.6 (Driver: 13.4.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.27.0)
I0000 00:00:1795103678.601996   16950 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1

  validation score -1.5495 (best -1.5495; higher is better)
300/300 - 65s - 217ms/step - loss: 2.2794
Epoch 2/3
  validation score -1.7056 (best -1.5495; higher is better)
300/300 - 19s - 62ms/step - loss: 1.7324
Epoch 3/3
  validation score -1.6429 (best -1.5495; higher is better)
300/300 - 19s - 63ms/step - loss: 1.5655
stage 2: fine-tune 66 backbone layers from block_6_expand
Epoch 1/30


I0000 00:00:1795103791.994418   16950 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_30160__.191


  validation score -1.5767 (best -1.5495; higher is better)
300/300 - 60s - 198ms/step - loss: 1.5879
Epoch 2/30
  validation score -2.0399 (best -1.5495; higher is better)
300/300 - 29s - 95ms/step - loss: 1.3049
Epoch 3/30
  validation score -1.2659 (best -1.2659; higher is better)
300/300 - 29s - 95ms/step - loss: 1.0795
Epoch 4/30
  validation score -1.1416 (best -1.1416; higher is better)
300/300 - 29s - 95ms/step - loss: 0.9648
Epoch 5/30
  validation score -1.3051 (best -1.1416; higher is better)
300/300 - 25s - 82ms/step - loss: 0.8993
Epoch 6/30
  validation score -1.0514 (best -1.0514; higher is better)
300/300 - 24s - 81ms/step - loss: 0.8458
Epoch 7/30
  validation score -1.1779 (best -1.0514; higher is better)
300/300 - 26s - 87ms/step - loss: 0.8005
Epoch 8/30
  validation score -1.2408 (best -1.0514; higher is better)
300/300 - 23s - 76ms/step - loss: 0.7515
Epoch 9/30
  validation score -1.5200 (best -1.0514; higher is better)
300/300 - 18s - 61ms/step - loss: 0.7198
Ep

... (20 outputs left out)

INFO:tensorflow:Assets written to: /tmp/tmpcs6ylwns/assets


Saved artifact at '/tmp/tmpcs6ylwns'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='image')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  125283467493840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125283467496912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125283467496720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125283467494800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125283467498064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125283467495376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125283467497488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125283467493648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125283467495184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125283467499024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125283467497680: Ten

/home/joe/venvs/ESPclassifier/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1795104651.518583   16832 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1795104651.518639   16832 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1795104651.518796   16832 reader.cc:83] Reading SavedModel from: /tmp/tmpcs6ylwns
I0000 00:00:1795104651.530898   16832 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1795104651.530930   16832 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpcs6ylwns
I0000 00:00:1795104651.627830   16832 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1795104652.128287   16832 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpcs6ylwns
I0000 00:00:1795104652.245888   16832 loader.cc:471] SavedModel load for ta

esp32_only: best validation balanced log-loss 1.1942 after 25 epochs, 8.2 min
selected species arm: esp32_plus_outside


## 11. Evaluation: per crop, per visit, and the automation rate

Each arm's uint8 model scores every validation and test crop. **Frame metrics** count each crop separately. **Visit metrics** average the probabilities of all `chain` crops in a (visit, species) track; this is the hub's job.

The automation rule is chosen on validation tracks:
- **Species auto-label:** the top species has at least `MIN_TRAIN_VISITS_FOR_AUTO` camera training visits, and its averaged probability is at least `t_species`.
- **Group auto-label:** if the species isn't automatic, the top group's summed probability is at least `t_group`.
- **Review:** everything else.

Each threshold is the lowest one that keeps validation precision at or above `AUTO_TARGET_PRECISION`.


In [11]:
group_of = {s: g for g, members in SPECIES_GROUPS.items() for s in members}
allowed_auto = {s for s, n in train_visits_per_species.items() if n >= MIN_TRAIN_VISITS_FOR_AUTO}


def score_table(path, table):
    ds = make_eval_ds([records[i]["jpeg"] for i in table["index"]],
                      table[["left", "top", "right", "bottom"]].to_numpy(np.int32), SPECIES_IMG)
    return tflite_run(path, ds)


def frame_metrics(table, probs, classes):
    idx = {c: k for k, c in enumerate(classes)}
    truth = table["species"].map(idx).fillna(-1).to_numpy(int)
    pred = probs.argmax(axis=1)
    known = truth >= 0
    recalls = {classes[c]: float(np.mean(pred[truth == c] == c)) for c in np.unique(truth[known])}
    return {"crops": int(known.sum()), "top1": float(np.mean(pred[known] == truth[known])) if known.any() else np.nan,
            "macro_recall": float(np.mean(list(recalls.values()))) if recalls else np.nan}, recalls


def track_probs(table, probs):
    """Average chain-crop probabilities per (visit, labelled species)."""
    mask = table["kind"].eq("chain").to_numpy()
    sub = table[mask].reset_index(drop=True)
    p = probs[mask]
    rows, mats = [], []
    for (visit, species), g in sub.groupby(["visit_id", "species"]):
        rows.append({"visit_id": visit, "species": species, "frames": len(g)})
        mats.append(p[g.index.to_numpy()].mean(axis=0))
    return pd.DataFrame(rows), (np.stack(mats) if mats else np.zeros((0, probs.shape[1])))


def group_probs(p, classes):
    groups = sorted(set(group_of.get(c, c) for c in classes))
    gp = np.zeros((len(p), len(groups)))
    for k, c in enumerate(classes):
        gp[:, groups.index(group_of.get(c, c))] += p[:, k]
    return gp, groups


def choose_threshold(conf, correct, eligible):
    best = 1.01
    for t in sorted(set(np.round(conf[eligible], 3)), reverse=True):
        keep = eligible & (conf >= t)
        if keep.sum() >= 5 and correct[keep].mean() >= AUTO_TARGET_PRECISION:
            best = float(t)
        elif keep.sum() >= 5:
            break
    return best


def automation(tracks, p, classes, t_species, t_group):
    top = p.argmax(axis=1); conf = p.max(axis=1)
    pred = np.array([classes[k] for k in top])
    gp, groups = group_probs(p, classes)
    gpred = np.array([groups[k] for k in gp.argmax(axis=1)]); gconf = gp.max(axis=1)
    species_auto = np.isin(pred, list(allowed_auto)) & (conf >= t_species)
    group_auto = ~species_auto & (gconf >= t_group)
    truth = tracks["species"].to_numpy()
    out = tracks.assign(predicted=pred, confidence=conf, group_predicted=gpred, group_confidence=gconf,
                        decision=np.where(species_auto, "species", np.where(group_auto, "group", "review")))
    out["correct"] = np.where(species_auto, pred == truth,
                              np.where(group_auto, gpred == np.array([group_of.get(s, s) for s in truth]), np.nan))
    return out


eval_rows_out, per_class_rows, automation_rows = [], [], []
for arm, info in arm_results.items():
    classes = info["classes"]
    for split, table in (("validation", species_val), ("test", species_test)):
        _, probs, info["tensor_info"] = score_table(info["uint8"], table)
        info[f"{split}_probs"] = probs
        for kind in ("chain", "detector", "gt_box", "static_roi"):
            mask = table["kind"].eq(kind).to_numpy()
            if mask.any():
                m, recalls = frame_metrics(table[mask], probs[mask], classes)
                eval_rows_out.append({"arm": arm, "split": split, "unit": f"frame:{kind}", **m})
                if split == "test" and kind == "chain":
                    per_class_rows += [{"arm": arm, "species": c, "frame_recall": v} for c, v in recalls.items()]
        tracks, tp = track_probs(table, probs)
        info[f"{split}_tracks"] = (tracks, tp)
        if len(tracks):
            idx = {c: k for k, c in enumerate(classes)}
            truth = tracks["species"].map(idx).fillna(-1).to_numpy(int)
            hit = tp.argmax(axis=1) == truth
            recalls = [hit[truth == c].mean() for c in np.unique(truth[truth >= 0])]
            eval_rows_out.append({"arm": arm, "split": split, "unit": "visit_track", "crops": len(tracks),
                                  "top1": float(hit.mean()), "macro_recall": float(np.mean(recalls))})
    vt, vp = info["validation_tracks"]
    pred = np.array([classes[k] for k in vp.argmax(axis=1)]) if len(vt) else np.array([])
    info["t_species"] = choose_threshold(vp.max(axis=1), pred == vt["species"].to_numpy(), np.isin(pred, list(allowed_auto))) if len(vt) else 1.01
    gp, groups = group_probs(vp, classes) if len(vt) else (np.zeros((0, 1)), [])
    gtruth = np.array([group_of.get(s, s) for s in vt["species"]])
    gpred = np.array([groups[k] for k in gp.argmax(axis=1)]) if len(vt) else np.array([])
    info["t_group"] = choose_threshold(gp.max(axis=1), gpred == gtruth, np.ones(len(vt), bool)) if len(vt) else 1.01
    for split in ("validation", "test"):
        tracks, tp = info[f"{split}_tracks"]
        if not len(tracks):
            continue
        decided = automation(tracks, tp, classes, info["t_species"], info["t_group"])
        info[f"{split}_decisions"] = decided
        for decision in ("species", "group", "review"):
            part = decided[decided["decision"].eq(decision)]
            automation_rows.append({"arm": arm, "split": split, "decision": decision, "tracks": len(part),
                                    "share": len(part) / len(decided),
                                    "precision": float(part["correct"].mean()) if decision != "review" and len(part) else np.nan})

species_eval = pd.DataFrame(eval_rows_out); species_eval.to_csv(RUN_DIR / "results.csv", index=False)
automation_table = pd.DataFrame(automation_rows); automation_table.to_csv(RUN_DIR / "automation.csv", index=False)
pd.DataFrame(per_class_rows).to_csv(RUN_DIR / "per_class_test_recall.csv", index=False)
display(species_eval.round(3))
print("automation (share of visit tracks, and precision of the automatic labels):")
display(automation_table.round(3))
for arm, info in arm_results.items():
    print(f"{arm}: t_species={info['t_species']:.3f}  t_group={info['t_group']:.3f}")

# Re-select the arm on what the hub needs: the share of validation tracks it may label automatically
# (species or group, at the target precision); balanced log-loss only breaks ties.
def auto_share(info):
    decided = info.get("validation_decisions")
    return 0.0 if decided is None else float((decided["decision"] != "review").mean())


previous_choice = best_arm
best_arm = max(arm_results, key=lambda a: (auto_share(arm_results[a]), -arm_results[a]["best_val_balanced_nll"]))
for arm, info in arm_results.items():
    print(f"{arm}: validation auto-labelled share {auto_share(info):.1%}, balanced log-loss {info['best_val_balanced_nll']:.3f}")
print(f"selected species arm: {best_arm}" + (f" (log-loss alone picked {previous_choice})" if best_arm != previous_choice else ""))


/home/joe/venvs/ESPclassifier/lib/python3.12/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
/home/joe/venvs/ESPclassifier/lib/python3.12/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
/home/joe/venvs/ESPclassifier/lib/python3.12/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated 

,arm,split,unit,crops,top1,macro_recall
0,esp32_plus_outside,validation,frame:chain,1545,0.590,0.567
1,esp32_plus_outside,validation,frame:detector,1319,0.622,0.599
2,esp32_plus_outside,validation,frame:gt_box,1655,0.599,0.566
3,esp32_plus_outside,validation,frame:static_roi,1018,0.672,0.330
4,esp32_plus_outside,validation,visit_track,26,0.654,0.547
5,esp32_plus_outside,test,frame:chain,408,0.777,0.662
6,esp32_plus_outside,test,frame:detector,368,0.758,0.659
7,esp32_plus_outside,test,frame:gt_box,420,0.760,0.582
8,esp32_plus_outside,test,frame:static_roi,299,0.873,0.501
9,esp32_plus_outside,test,visit_track,22,0.727,0.705


automation (share of visit tracks, and precision of the automatic labels):


,arm,split,decision,tracks,share,precision
0,esp32_plus_outside,validation,species,0,0.000,NaN
1,esp32_plus_outside,validation,group,0,0.000,NaN
2,esp32_plus_outside,validation,review,26,1.000,NaN
3,esp32_plus_outside,test,species,0,0.000,NaN
4,esp32_plus_outside,test,group,0,0.000,NaN
5,esp32_plus_outside,test,review,22,1.000,NaN
6,esp32_only,validation,species,5,0.192,1.000
7,esp32_only,validation,group,3,0.115,1.000
8,esp32_only,validation,review,18,0.692,NaN
9,esp32_only,test,species,6,0.273,1.000


esp32_plus_outside: t_species=1.010  t_group=1.010
esp32_only: t_species=0.682  t_group=0.704


## 12. Selected model: files, contract, confusion, review queue, comparison with v4

`review_queue.csv` lists the visits to look at first: every (visit, labelled species) track, train included, where the model's averaged answer disagrees with the label at a probability of at least `REVIEW_DISAGREEMENT_MIN_PROB`. Training tracks the model still confidently rejects are the strongest label-error candidates.


In [12]:
import shutil
best = arm_results[best_arm]
classes = best["classes"]
for src, name in ((best["uint8"], "model_uint8.tflite"), (best["float"], "model_float32.tflite"), (best["dir"] / "labels.txt", "labels.txt")):
    shutil.copyfile(src, RUN_DIR / name)

species_contract = {
    "model_file": "model_uint8.tflite", "float_model_file": "model_float32.tflite", "labels_file": "labels.txt",
    "task": "species of one localised bird", "arm": best_arm, "classes": classes,
    "camera_trained_classes": camera_train_species, "outside_only_classes": sorted(set(classes) - set(camera_train_species)),
    "input_shape": best["tensor_info"]["input_shape"], "input_dtype": best["tensor_info"]["input_dtype"],
    "input_quantization": best["tensor_info"]["input_quantization"],
    "output_dtype": best["tensor_info"]["output_dtype"], "output_quantization": best["tensor_info"]["output_quantization"],
    "output": "softmax probabilities in labels.txt order",
    "geometry": "detector_box_square_context_crop_then_letterbox", "geometry_location": "outside the model",
    "box_source": "detector detection box in original frame pixels (fallback: the isBird selected crop)",
    "crop_rule": {"context": SPECIES_CONTEXT, "min_size_px": SPECIES_MIN_CROP, "shape": "square, centred, shifted inside the frame"},
    "color_order": "RGB", "jpeg_decode_channels": 3, "resize_method": "TensorFlow bilinear", "resize_antialias": True,
    "letterbox_padding_value": 0, "numeric_scaling": "[0,255] to [-1,1] inside the model",
    "visit_aggregation": {"gap_seconds": VISIT_GAP_SECONDS, "rule": "mean probability over the visit's crops"},
    "automation": {"species_min_confidence": best["t_species"], "group_min_confidence": best["t_group"],
                   "auto_species": sorted(allowed_auto & set(classes)), "groups": SPECIES_GROUPS,
                   "target_precision": AUTO_TARGET_PRECISION},
    "localizer": chosen_localizer, "localizer_model_sha256_prefix": det_sha,
}
write_json(RUN_DIR / "species_contract.json", species_contract)

# confusion: test chain crops, frame level
table, probs = species_test, best["test_probs"]
mask = table["kind"].eq("chain").to_numpy() & table["species"].isin(classes).to_numpy()
if mask.any():
    present = sorted(set(table["species"][mask]))
    matrix = pd.crosstab(pd.Categorical(table["species"][mask], categories=present),
                         pd.Series([classes[k] for k in probs[mask].argmax(axis=1)], name="predicted"),
                         rownames=["true"], dropna=False)
    matrix = matrix.loc[:, matrix.sum(axis=0) > 0]
    matrix.to_csv(RUN_DIR / "test_confusion_chain.csv")
    display(matrix)

# review queue over every track (train included)
all_rows = []
for row in boxes.rename(columns={"index": "record_idx"}).itertuples(index=False):
    r = records[row.record_idx]
    if (row.record_idx, row.box) in det_box_of:
        crop = species_crop_bounds(det_box_of[(row.record_idx, row.box)][0], r["width"], r["height"])
    elif row.in_pi_crop and r["roi"] is not None:
        crop = pi_crop(r)
    else:
        crop = species_crop_bounds(r["boxes"][row.box], r["width"], r["height"])
    all_rows.append({"index": row.record_idx, "box": row.box, "sample_id": r["sample_id"], "visit_id": row.visit_id,
                     "species": row.species, "split": row.split, "kind": "chain",
                     "left": crop[0], "top": crop[1], "right": crop[2], "bottom": crop[3]})
all_table = pd.DataFrame(all_rows)
_, all_probs, _ = score_table(RUN_DIR / "model_uint8.tflite", all_table)
tracks, tp = track_probs(all_table, all_probs)
tracks["split"] = tracks["visit_id"].map(visit_split).fillna("train")
tracks["predicted"] = [classes[k] for k in tp.argmax(axis=1)]
tracks["predicted_prob"] = tp.max(axis=1)
tracks["label_prob"] = [tp[n, classes.index(s)] if s in classes else np.nan for n, s in enumerate(tracks["species"])]
first = all_table.groupby(["visit_id", "species"])["sample_id"].agg(lambda s: " ".join(list(s)[:3]))
tracks["example_frames"] = [first[(v, s)] for v, s in zip(tracks["visit_id"], tracks["species"])]
queue = tracks[(tracks["predicted"] != tracks["species"]) & (tracks["predicted_prob"] >= REVIEW_DISAGREEMENT_MIN_PROB)]
queue = queue.sort_values(["split", "predicted_prob"], ascending=[False, False])
queue.to_csv(RUN_DIR / "review_queue.csv", index=False)
tracks.to_csv(RUN_DIR / "all_track_predictions.csv", index=False)
print(f"review queue: {len(queue)} of {len(tracks)} tracks ({(queue['split'] == 'train').sum()} from training visits)")
display(queue.head(25))

# previous (v4) species model on the same test chain crops
comparison = []
try:
    prev_dir = portable_path(PREVIOUS_SPECIES_DIR)
    prev_classes = [l.strip() for l in (prev_dir / "labels.txt").read_text(encoding="utf-8").splitlines() if l.strip()]
    chain_test = species_test[species_test["kind"].eq("chain")].reset_index(drop=True)
    _, prev_probs, _ = score_table(prev_dir / "model_uint8.tflite", chain_test)
    m, _ = frame_metrics(chain_test, prev_probs, prev_classes)
    comparison.append({"model": "v4 species", "unit": "frame:chain", **m})
except Exception as error:
    print("v4 comparison skipped:", repr(error))
chain_mask = species_test["kind"].eq("chain").to_numpy()
m, _ = frame_metrics(species_test[chain_mask], best["test_probs"][chain_mask], classes)
comparison.append({"model": f"v4.1 species ({best_arm})", "unit": "frame:chain", **m})
comparison = pd.DataFrame(comparison); comparison.to_csv(RUN_DIR / "v4_vs_v4_1.csv", index=False)
display(comparison.round(3))


predicted,American Crow,American Robin,Black-capped Chickadee,Blue Jay,Common Grackle,Dark-eyed Junco,Downy Woodpecker,House Sparrow,Mourning Dove,Northern Cardinal,Song Sparrow,Tufted Titmouse,White-breasted Nuthatch
true,,,,,,,,,,,,,
American Crow,72,0,0,3,1,0,0,1,1,0,0,0,0
Blue Jay,0,0,0,7,0,0,11,5,1,0,0,2,0
House Sparrow,2,1,6,3,8,1,2,227,3,0,4,2,18
Mourning Dove,0,0,0,1,0,0,1,3,0,0,8,0,0
Northern Cardinal,0,0,0,0,0,0,0,0,0,2,0,0,0
Song Sparrow,0,0,0,0,0,0,0,0,0,0,4,0,0
White-breasted Nuthatch,0,0,0,0,0,0,0,1,0,0,2,0,5


/home/joe/venvs/ESPclassifier/lib/python3.12/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


review queue: 8 of 144 tracks (2 from training visits)


,visit_id,species,frames,split,predicted,predicted_prob,label_prob,example_frames
135,legacy/capture_20261014T215358_419298Z_e05bba42,Mourning Dove,3,validation,Blue Jay,0.774740,0.001302,capture_20261014T215410_546302Z_69313778 captu...
48,esp32cam-b8aa6d0d5020/capture_20261028T191144_...,Black-capped Chickadee,1,validation,Blue Jay,0.507812,0.066406,capture_20261028T191144_602119Z_b6c11a79
52,esp32cam-b8aa6d0d5020/capture_20261028T194656_...,Black-capped Chickadee,7,train,House Sparrow,0.983259,0.000000,capture_20261028T195047_538060Z_91de0626 captu...
36,esp32cam-b8aa6d0d5020/capture_20261028T154956_...,Black-capped Chickadee,1,train,House Sparrow,0.554688,0.027344,capture_20261028T155029_947348Z_c257d8a6
109,legacy/capture_20261007T170625_439703Z_a862e443,Tufted Titmouse,1,test,Song Sparrow,0.937500,0.003906,capture_20261007T170625_439703Z_a862e443
102,legacy/capture_20261007T164032_688006Z_b30dd1dd,House Sparrow,2,test,Song Sparrow,0.791016,0.068359,capture_20261007T164243_163703Z_108bac7b captu...
103,legacy/capture_20261007T164032_688006Z_b30dd1dd,White-breasted Nuthatch,2,test,Song Sparrow,0.675781,0.005859,capture_20261007T164032_688006Z_b30dd1dd captu...
57,esp32cam-b8aa6d0d5020/capture_20261116T152338_...,House Sparrow,1,test,White-breasted Nuthatch,0.593750,0.109375,capture_20261116T152338_969077Z_df9f22e1


/home/joe/venvs/ESPclassifier/lib/python3.12/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


,model,unit,crops,top1,macro_recall
0,v4 species,frame:chain,408,0.676,0.704
1,v4.1 species (esp32_plus_outside),frame:chain,408,0.777,0.662


## 13. Summary and deployment notes


In [13]:
selected_auto = automation_table[automation_table["arm"].eq(best_arm) & automation_table["split"].eq("test")]
summary = {
    "run": RUN_NAME, "created_utc": created_utc.isoformat(), "quick_run": QUICK_RUN,
    "images": len(records), "visits": len(visits), "species_tracks": int(boxes.groupby(["visit_id", "species"]).ngroups),
    "localizer": chosen_localizer, "localisation_rates": match_table.loc["ALL"].to_dict(),
    "selected_arm": best_arm,
    "arms": {a: {k: i[k] for k in ("best_val_balanced_nll", "epochs", "minutes", "t_species", "t_group")} | {"classes": len(i["classes"])}
             for a, i in arm_results.items()},
    "test": species_eval[species_eval["arm"].eq(best_arm) & species_eval["split"].eq("test")].to_dict("records"),
    "test_automation": selected_auto.to_dict("records"),
    "config": {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, bool, list, tuple, dict))},
}
write_json(RUN_DIR / "run_summary.json", summary)
fmt = lambda t: "never (always review)" if t > 1 else f"{t:.3f}"
deploy = f"""# Deploying {RUN_NAME}

Files: `model_uint8.tflite` (or `model_float32.tflite`), `labels.txt`, `species_contract.json`. Keep all three
in one folder; the hub reads the contract and labels from beside the model and refuses to start if they disagree.

Hub command (server/raspberry_pi_server.py):

    --is-bird-model   <isbird run>/isbird/model_uint8.tflite     (inference_contract.json + threshold.json beside it)
    --localizer-model <v3 run>/selected_model/esp32_detector_int8.tflite   (esp32_detector_contract.json beside it)
    --species-model   <this run>/model_uint8.tflite

Localiser chosen by this run: {chosen_localizer} (see localisation_by_species.csv). If that is pi_detector, run the
hub with the detector as the gate instead (no --is-bird-model / --localizer-model); species then uses its boxes.

Per capture the hub localises birds on the selected crop, crops each box with the same rule as training
(square, side = {SPECIES_CONTEXT} x long side, min {SPECIES_MIN_CROP} px, shifted inside the frame), letterboxes on black to
{SPECIES_IMG} and stores per-box probabilities under `species` in the .inference.json. With no box it uses the selected
crop and records `localised: false`.

Visits (same camera, gap < {VISIT_GAP_SECONDS}s; start the hub with --visit-gap-seconds {VISIT_GAP_SECONDS}) average the
per-crop probabilities; closed visits are appended to visits.jsonl with the decision:
- species auto-label if the top species is in {sorted(allowed_auto & set(classes))} and its mean probability >= {fmt(best['t_species'])}
- otherwise group auto-label if the top group's summed probability >= {fmt(best['t_group'])}
- otherwise review.
Rebuild visits for a capture folder at any time: python -m server.visits rebuild <folder> --species-contract species_contract.json
"""
(RUN_DIR / "DEPLOY.md").write_text(deploy, encoding="utf-8")
print(deploy)
display(selected_auto.round(3))


# Deploying esp32_species_v4_1_20261119T155021Z

Files: `model_uint8.tflite` (or `model_float32.tflite`), `labels.txt`, `species_contract.json`. Keep all three
in one folder; the hub reads the contract and labels from beside the model and refuses to start if they disagree.

Hub command (server/raspberry_pi_server.py):

    --is-bird-model   <isbird run>/isbird/model_uint8.tflite     (inference_contract.json + threshold.json beside it)
    --localizer-model <v3 run>/selected_model/esp32_detector_int8.tflite   (esp32_detector_contract.json beside it)
    --species-model   <this run>/model_uint8.tflite

Localiser chosen by this run: esp32_roi_v3 (see localisation_by_species.csv). If that is pi_detector, run the
hub with the detector as the gate instead (no --is-bird-model / --localizer-model); species then uses its boxes.

Per capture the hub localises birds on the selected crop, crops each box with the same rule as training
(square, side = 1.5 x long side, min 48 px, shifted inside the f

,arm,split,decision,tracks,share,precision
3,esp32_plus_outside,test,species,0,0.0,NaN
4,esp32_plus_outside,test,group,0,0.0,NaN
5,esp32_plus_outside,test,review,22,1.0,NaN
